<a href="https://colab.research.google.com/github/luciasmolinas-glitch/University_Test/blob/main/Clinical_Mortality_MLOps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import shutil

# Safely locate the dataset path (handling suffixes and Drive locations)
possible_paths = [
    "/content/synthetic_clinical_dataset.csv",
    "/content/drive/MyDrive/AINT602_mortality_pipeline/data/raw/synthetic_clinical_dataset.csv"
]

dataset_path = None
for path in possible_paths:
    if os.path.exists(path):
        dataset_path = path
        break

# If it is only on Drive, copy it to /content/ for easy access
if dataset_path == "/content/drive/MyDrive/AINT602_mortality_pipeline/data/raw/synthetic_clinical_dataset.csv":
    try:
        shutil.copy(dataset_path, "/content/synthetic_clinical_dataset.csv")
        dataset_path = "/content/synthetic_clinical_dataset.csv"
        print("Successfully copied dataset from Google Drive to local runtime.")
    except Exception as e:
        print(f"Warning: Could not copy dataset from Drive: {e}")

if not dataset_path or not os.path.exists(dataset_path):
    raise FileNotFoundError("Please upload the synthetic_clinical_dataset.csv to the /content/ directory or mount your Google Drive!")

# Load dataset
df = pd.read_csv(dataset_path, sep=",")
print(f"Dataset loaded successfully from: {dataset_path}")
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

Dataset loaded successfully from: /content/synthetic_clinical_dataset.csv
Total Rows: 10000
Total Columns: 14


In [ ]:
# Create project repository structure and initial modules
import os

# Create directory structure
os.makedirs("src", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs("saved_model", exist_ok=True)
os.makedirs("sample_inputs", exist_ok=True)

print("Directory structure initialized successfully:")
print("- src/")
print("- tests/")
print("- saved_model/")
print("- sample_inputs/")

Directory structure initialized successfully:
- src/
- tests/
- saved_model/
- sample_inputs/


In [ ]:
%%writefile src/ingest.py
import pandas as pd
import os
import sys

def load_data(filepath="synthetic_clinical_dataset.csv"):
    if not os.path.exists(filepath):
        # Check alternate Colab location
        alternate = os.path.join("/content", filepath)
        if os.path.exists(alternate):
            filepath = alternate
        else:
            raise FileNotFoundError(f"Dataset not found at: {filepath}")

    df = pd.read_csv(filepath, sep=",")
    print(f"[Ingestion] Successfully ingested {len(df)} rows and {df.shape[1]} columns.")
    return df

if __name__ == "__main__":
    try:
        load_data()
    except Exception as e:
        print(f"Error: {e}")
        sys.exit(1)

Overwriting src/ingest.py


In [ ]:
%%writefile src/validate.py
import pandas as pd
import sys
from src.ingest import load_data

def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Age filtering: Must be an adult (18 <= age <= 120)
    initial_count = len(df)
    df_adult = df[(df["age"] >= 18) & (df["age"] <= 120)].copy()
    age_excluded = initial_count - len(df_adult)

    # 2. Physiological BP check: diastolic must be strictly less than systolic
    invalid_bp = df_adult["diastolic_bp"] >= df_adult["systolic_bp"]
    bp_excluded = invalid_bp.sum()
    df_clean = df_adult[~invalid_bp].copy()

    print(f"[Validation] Original rows: {initial_count}")
    print(f"[Validation] Excluded (Non-Adult): {age_excluded}")
    print(f"[Validation] Excluded (Invalid BP): {bp_excluded}")
    print(f"[Validation] Fully validated rows: {len(df_clean)}")
    return df_clean

if __name__ == "__main__":
    try:
        raw_data = load_data()
        validate_data(raw_data)
    except Exception as e:
        print(f"Error during validation: {e}")
        sys.exit(1)

Overwriting src/validate.py


In [ ]:
%%writefile src/features.py
import pandas as pd
import sys
from src.ingest import load_data

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Feature 1: Pulse Pressure
    df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]

    # Feature 2: Comorbidity Count
    df["comorbidity_count"] = df["diabetes"] + df["hypertension"]

    print(f"[Features] Successfully added engineered features. Columns: {list(df.columns[-2:])}")
    return df

if __name__ == "__main__":
    from src.validate import validate_data
    try:
        raw_data = load_data()
        valid_data = validate_data(raw_data)
        engineer_features(valid_data)
    except Exception as e:
        print(f"Error engineering features: {e}")
        sys.exit(1)

Overwriting src/features.py


In [ ]:
%%writefile src/train.py
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

from src.ingest import load_data
from src.validate import validate_data

def train_pipeline():
    # 1. Ingestion
    raw_df = load_data()

    # 2. Validation
    clean_df = validate_data(raw_df)

    # 3. Feature Engineering
    from src.features import engineer_features
    processed_df = engineer_features(clean_df)

    # 4. Prepare training set
    features = [
        "age", "bmi", "systolic_bp", "diastolic_bp",
        "glucose", "cholesterol", "creatinine",
        "diabetes", "hypertension", "pulse_pressure", "comorbidity_count"
    ]
    X = processed_df[features]
    y = processed_df["mortality"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 5. Train model
    print("[Model Training] Fitting Random Forest model...")
    model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
    model.fit(X_train, y_train)

    # 6. Evaluation
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, probs)
    print("\n=== Test Performance Evaluation ===")
    print(classification_report(y_test, preds))
    print(f"ROC-AUC Metric Score: {auc:.4f}")

    # Save metrics to evaluate results file
    os.makedirs("saved_model", exist_ok=True)
    with open("saved_model/evaluation_results.txt", "w") as f:
        f.write("=== EVALUATION METRICS ===\n")
        f.write(f"ROC-AUC: {auc:.4f}\n\n")
        f.write(classification_report(y_test, preds))

    # Save training metrics to artifacts
    with open("saved_model/model.pkl", "wb") as f:
        pickle.dump(model, f)
    print("[Model Training] Model successfully saved to saved_model/model.pkl")

if __name__ == "__main__":
    train_pipeline()

Overwriting src/train.py


In [ ]:
%%writefile src/predict.py
import argparse
import pandas as pd
import pickle
import os
import sys

def run_prediction(input_csv, output_csv="predictions.csv"):
    model_path = "saved_model/model.pkl"
    if not os.path.exists(model_path):
        print(f"Error: Trained model file not found at {model_path}. Run training first.")
        sys.exit(1)

    if not os.path.exists(input_csv):
        print(f"Error: Sample input CSV not found at {input_csv}")
        sys.exit(1)

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    # Ingest clinical samples
    df = pd.read_csv(input_csv)
    original_df = df.copy()

    # Feature alignment and parsing
    # Ensure features exist or populate engineered metrics
    if "pulse_pressure" not in df.columns:
        df["pulse_pressure"] = df["systolic_bp"] - df["diastolic_bp"]
    if "comorbidity_count" not in df.columns:
        df["comorbidity_count"] = df["diabetes"] + df["hypertension"]

    features = [
        "age", "bmi", "systolic_bp", "diastolic_bp",
        "glucose", "cholesterol", "creatinine",
        "diabetes", "hypertension", "pulse_pressure", "comorbidity_count"
    ]

    # Predict mortality risk
    X = df[features]
    original_df["mortality_prediction"] = model.predict(X)
    original_df["mortality_probability"] = model.predict_proba(X)[:, 1]

    # Save and output sample prediction predictions
    original_df.to_csv(output_csv, index=False)
    print(f"\nPredictions completed. Output saved to: {output_csv}")
    print(original_df[["patient_id", "mortality_prediction", "mortality_probability"]].head())

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="CLI to predict Patient Mortality Risk.")
    parser.add_argument("--input", type=str, required=True, help="Path to the patient evaluation CSV file.")
    args = parser.parse_args()
    run_prediction(args.input)

Overwriting src/predict.py


In [ ]:
%%writefile sample_inputs/new_patients.csv
patient_id,age,sex,bmi,systolic_bp,diastolic_bp,glucose,cholesterol,creatinine,diabetes,hypertension,diagnosis,readmission_30d
15001,65,Male,26.5,145,95,140,220,1.4,1,1,Normal,0
15002,45,Female,22.1,118,78,92,185,0.8,0,0,Pneumonia,0
15003,78,Male,31.2,158,102,185,250,2.1,1,1,Myocardial Infarction,1

Overwriting sample_inputs/new_patients.csv


In [ ]:
%%writefile tests/test_pipeline.py
import pytest
import pandas as pd
import os
from src.validate import validate_data
from src.features import engineer_features

def test_validation_rules():
    # Test record data validation logic
    raw_test_data = pd.DataFrame({
        "patient_id": [1, 2, 3],
        "age": [15, 45, 125],
        "systolic_bp": [120, 115, 140],
        "diastolic_bp": [80, 120, 90]
    })

    validated = validate_data(raw_test_data)

    # ID 1 (under 18) should be removed
    # ID 2 (diastolic >= systolic) should be removed
    # ID 3 (age 125 > 120 limit) should be removed
    assert len(validated) == 0

def test_feature_creation():
    sample_df = pd.DataFrame({
        "systolic_bp": [130],
        "diastolic_bp": [80],
        "diabetes": [1],
        "hypertension": [1]
    })

    engineered = engineer_features(sample_df)
    assert engineered.iloc[0]["pulse_pressure"] == 50
    assert engineered.iloc[0]["comorbidity_count"] == 2

Overwriting tests/test_pipeline.py


In [ ]:
%%writefile requirements.txt
pytest
pandas
numpy
scikit-learn

Overwriting requirements.txt


In [ ]:
import zipfile
import os

def zip_repository(zip_name="patient_mortality_repository.zip"):
    targets = [
        "src",
        "tests",
        "saved_model",
        "sample_inputs",
        "requirements.txt",
        "README.md"
    ]

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for target in targets:
            if os.path.exists(target):
                if os.path.isdir(target):
                    for root, dirs, filenames in os.walk(target):
                        for filename in filenames:
                            filepath = os.path.join(root, filename)
                            arcname = os.path.relpath(filepath, os.getcwd())
                            zipf.write(filepath, arcname)
                else:
                    zipf.write(target, target)

    print(f"Success! Compressed all repository items into: {os.path.abspath(zip_name)}")
    print("You can now download this file from the left files explorer panel!")

zip_repository()

Success! Compressed all repository items into: /content/patient_mortality_repository.zip
You can now download this file from the left files explorer panel!


In [ ]:
import zipfile
import os

def create_archive(zip_name="patient_mortality_repository.zip"):
    targets = [
        "src",
        "tests",
        "saved_model",
        "sample_inputs",
        "requirements.txt",
        "README.md"
    ]

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for target in targets:
            if os.path.exists(target):
                if os.path.isdir(target):
                    for root, dirs, filenames in os.walk(target):
                        for filename in filenames:
                            filepath = os.path.join(root, filename)
                            arcname = os.path.relpath(filepath, os.getcwd())
                            zipf.write(filepath, arcname)
                else:
                    zipf.write(target, target)

    print(f"Success! Archive created at: {os.path.abspath(zip_name)}")
    print("Click the circular Refresh icon at the top of the Files menu on the left side of your screen to see it!")

create_archive()

Success! Archive created at: /content/patient_mortality_repository.zip
Click the circular Refresh icon at the top of the Files menu on the left side of your screen to see it!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# Check missing values in each column

missing_values = df.isnull().sum()

print("MISSING VALUES REPORT")
print("---------------------")
print(missing_values)


MISSING VALUES REPORT
---------------------
patient_id         0
age                0
sex                0
bmi                0
systolic_bp        0
diastolic_bp       0
glucose            0
cholesterol        0
creatinine         0
diabetes           0
hypertension       0
diagnosis          0
readmission_30d    0
mortality          0
dtype: int64


In [ ]:

# Check duplicate records

duplicate_rows = df.duplicated().sum()

# Check repeated patient IDs

duplicate_patient_ids = df["patient_id"].duplicated().sum()

# Display results

print("DUPLICATE VALIDATION REPORT")
print("---------------------------")
print("Duplicate Rows:", duplicate_rows)
print("Duplicate Patient IDs:", duplicate_patient_ids)


DUPLICATE VALIDATION REPORT
---------------------------
Duplicate Rows: 0
Duplicate Patient IDs: 0


In [ ]:
# Age validation

print("AGE VALIDATION REPORT")
print("---------------------")

print("Minimum Age:", df["age"].min())
print("Maximum Age:", df["age"].max())


AGE VALIDATION REPORT
---------------------
Minimum Age: 0
Maximum Age: 90


In [ ]:
# Check for invalid or implausible ages

invalid_ages = (
    (df["age"] < 0) |
    (df["age"] > 120)
).sum()

print("Invalid Ages:", invalid_ages)


Invalid Ages: 0


In [ ]:
# Identify adult patients

adult_count = (df["age"] >= 18).sum()

under_18_count = (df["age"] < 18).sum()

print("Adult Patients (18+):", adult_count)
print("Patients Under 18:", under_18_count)


Adult Patients (18+): 9796
Patients Under 18: 204


In [ ]:
# Create adult-only dataset

adult_df = df[
    (df["age"] >= 18) &
    (df["age"] <= 120)
].copy()

print("ADULT DATASET CREATED")
print("---------------------")
print("Original Records:", len(df))
print("Adult Records:", len(adult_df))
print("Excluded Records:", len(df) - len(adult_df))


ADULT DATASET CREATED
---------------------
Original Records: 10000
Adult Records: 9796
Excluded Records: 204


In [ ]:
# BMI Validation

print("BMI VALIDATION REPORT")
print("---------------------")

print("Minimum BMI:", adult_df["bmi"].min())
print("Maximum BMI:", adult_df["bmi"].max())
print("Average BMI:", adult_df["bmi"].mean().round(2))


BMI VALIDATION REPORT
---------------------
Minimum BMI: 15.0
Maximum BMI: 50.0
Average BMI: 27.02


In [ ]:
# Blood Pressure Validation

print("BLOOD PRESSURE VALIDATION")
print("-------------------------")

print("Minimum Systolic BP:", adult_df["systolic_bp"].min())
print("Maximum Systolic BP:", adult_df["systolic_bp"].max())

print("Minimum Diastolic BP:", adult_df["diastolic_bp"].min())
print("Maximum Diastolic BP:", adult_df["diastolic_bp"].max())


BLOOD PRESSURE VALIDATION
-------------------------
Minimum Systolic BP: 90.0
Maximum Systolic BP: 180.0
Minimum Diastolic BP: 60.0
Maximum Diastolic BP: 110.0


In [ ]:
# Check blood pressure consistency

invalid_bp = (
    adult_df["diastolic_bp"] >= adult_df["systolic_bp"]
).sum()

print("BLOOD PRESSURE CONSISTENCY REPORT")
print("---------------------------------")
print("Invalid BP Records:", invalid_bp)


BLOOD PRESSURE CONSISTENCY REPORT
---------------------------------
Invalid BP Records: 156


In [ ]:
# Display suspicious blood pressure records

invalid_bp_records = adult_df[
    adult_df["diastolic_bp"] >= adult_df["systolic_bp"]
]

invalid_bp_records[
    ["patient_id", "age", "systolic_bp", "diastolic_bp"]
].head(10)


,patient_id,age,systolic_bp,diastolic_bp
15,1016,44,99.0,108.0
21,1022,50,90.0,97.0
35,1036,33,91.0,91.0
64,1065,69,94.0,98.0
111,1112,54,95.0,95.0
267,1268,29,97.0,98.0
334,1335,72,98.0,99.0
339,1340,44,90.0,92.0
492,1493,56,94.0,95.0
541,1542,34,106.0,110.0


In [ ]:
# Flag inconsistent blood pressure records

adult_df["invalid_bp_flag"] = (
    adult_df["diastolic_bp"] >= adult_df["systolic_bp"]
).astype(int)

print("BLOOD PRESSURE QUALITY FLAGS")
print("----------------------------")

print(adult_df["invalid_bp_flag"].value_counts())


BLOOD PRESSURE QUALITY FLAGS
----------------------------
invalid_bp_flag
0    9640
1     156
Name: count, dtype: int64


In [ ]:
# Glucose Validation

print("GLUCOSE VALIDATION REPORT")
print("-------------------------")

print("Minimum Glucose:", adult_df["glucose"].min())
print("Maximum Glucose:", adult_df["glucose"].max())
print("Average Glucose:", adult_df["glucose"].mean().round(2))


GLUCOSE VALIDATION REPORT
-------------------------
Minimum Glucose: 70.0
Maximum Glucose: 198.6
Average Glucose: 106.02


In [ ]:
# Cholesterol Validation

print("CHOLESTEROL VALIDATION REPORT")
print("-----------------------------")

print("Minimum Cholesterol:", adult_df["cholesterol"].min())
print("Maximum Cholesterol:", adult_df["cholesterol"].max())
print("Average Cholesterol:", adult_df["cholesterol"].mean().round(2))


CHOLESTEROL VALIDATION REPORT
-----------------------------
Minimum Cholesterol: 120.0
Maximum Cholesterol: 300.0
Average Cholesterol: 190.87


In [ ]:
# Creatinine Validation

print("CREATININE VALIDATION REPORT")
print("----------------------------")

print("Minimum Creatinine:", adult_df["creatinine"].min())
print("Maximum Creatinine:", adult_df["creatinine"].max())
print("Average Creatinine:", adult_df["creatinine"].mean().round(2))


CREATININE VALIDATION REPORT
----------------------------
Minimum Creatinine: 0.5
Maximum Creatinine: 2.13
Average Creatinine: 1.0


In [ ]:
# Mortality Target Validation

print("MORTALITY TARGET VALIDATION")
print("---------------------------")

print(adult_df["mortality"].value_counts(dropna=False))


MORTALITY TARGET VALIDATION
---------------------------
mortality
0    9529
1     267
Name: count, dtype: int64


In [ ]:
# Validate all categorical columns

categorical_columns = [
    "sex",
    "diabetes",
    "hypertension",
    "diagnosis",
    "readmission_30d",
    "mortality"
]

for column in categorical_columns:
    print(f"\n--- {column.upper()} ---")
    print(adult_df[column].value_counts(dropna=False))
    print("Unique Values:", adult_df[column].nunique(dropna=False))



--- SEX ---
sex
Female    4997
Male      4683
Other      116
Name: count, dtype: int64
Unique Values: 3

--- DIABETES ---
diabetes
0    8625
1    1171
Name: count, dtype: int64
Unique Values: 2

--- HYPERTENSION ---
hypertension
0    6817
1    2979
Name: count, dtype: int64
Unique Values: 2

--- DIAGNOSIS ---
diagnosis
Normal           5476
Pneumonia        1971
Heart Failure    1722
Sepsis            627
Name: count, dtype: int64
Unique Values: 4

--- READMISSION_30D ---
readmission_30d
0    8426
1    1370
Name: count, dtype: int64
Unique Values: 2

--- MORTALITY ---
mortality
0    9529
1     267
Name: count, dtype: int64
Unique Values: 2


In [ ]:
# Create a clean modelling dataset

clean_df = adult_df[
    adult_df["invalid_bp_flag"] == 0
].copy()

In [ ]:
# Remove the validation flag from model data
clean_df = clean_df.drop(columns=["invalid_bp_flag"])

print("DATA CLEANING REPORT")
print("--------------------")
print("Original Adult Records:", len(adult_df))
print("Clean Records:", len(clean_df))
print("Excluded BP Records:", len(adult_df) - len(clean_df))
print("Total Columns:", clean_df.shape[1])


DATA CLEANING REPORT
--------------------
Original Adult Records: 9796
Clean Records: 9640
Excluded BP Records: 156
Total Columns: 14


In [ ]:
# Feature Engineering: Pulse Pressure

clean_df["pulse_pressure"] = (
    clean_df["systolic_bp"] - clean_df["diastolic_bp"]
)

print("PULSE PRESSURE FEATURE CREATED")
print("------------------------------")

print(clean_df[
    ["systolic_bp", "diastolic_bp", "pulse_pressure"]
].head())

print("\nTotal Columns:", clean_df.shape[1])


PULSE PRESSURE FEATURE CREATED
------------------------------
   systolic_bp  diastolic_bp  pulse_pressure
0        131.0          81.0            50.0
1        118.0          91.0            27.0
2        144.0          73.0            71.0
3        112.0          76.0            36.0
4        102.0          65.0            37.0

Total Columns: 15


In [ ]:
# Feature Engineering: Comorbidity Count

clean_df["comorbidity_count"] = (
    clean_df["diabetes"] + clean_df["hypertension"]
)

print("COMORBIDITY FEATURE CREATED")
print("---------------------------")

print(clean_df[
    ["diabetes", "hypertension", "comorbidity_count"]
].head(10))

print("\nTotal Records:", clean_df.shape[0])
print("Total Columns:", clean_df.shape[1])


COMORBIDITY FEATURE CREATED
---------------------------
   diabetes  hypertension  comorbidity_count
0         0             0                  0
1         0             1                  1
2         0             0                  0
3         0             0                  0
4         0             0                  0
5         0             0                  0
6         0             1                  1
7         0             0                  0
8         0             0                  0
9         0             1                  1

Total Records: 9640
Total Columns: 16


In [ ]:
import zipfile
import os

def archive_project(zip_name="patient_mortality_repository.zip"):
    # Define the list of files and directories to include in the repository zip
    targets = [
        "src",
        "tests",
        "sample_inputs",
        "requirements.txt"
    ]

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for target in targets:
            if os.path.exists(target):
                if os.path.isdir(target):
                    for root, dirs, files in os.walk(target):
                        for file in files:
                            filepath = os.path.join(root, file)
                            # Calculate path relative to the directory containing targets
                            arcname = os.path.relpath(filepath, os.getcwd())
                            zipf.write(filepath, arcname)
                else:
                    zipf.write(target, target)

    if os.path.exists(zip_name):
        size_kb = os.path.getsize(zip_name) / 1024
        print(f"[Verification] SUCCESS: '{zip_name}' created successfully.")
        print(f"[Verification] Archive Size: {size_kb:.2f} KB")
        print("\nFolders and files packaged:")
        with zipfile.ZipFile(zip_name, 'r') as z:
            for name in z.namelist():
                print(f" - {name}")
    else:
        print("[Verification] ERROR: Archive creation failed.")

archive_project()

[Verification] SUCCESS: 'patient_mortality_repository.zip' created successfully.
[Verification] Archive Size: 4.33 KB

Folders and files packaged:
 - src/features.py
 - src/validate.py
 - src/predict.py
 - src/ingest.py
 - src/train.py
 - tests/test_pipeline.py
 - sample_inputs/new_patients.csv
 - requirements.txt


In [ ]:
from google.colab import files
import os

zip_path = "patient_mortality_repository.zip"
if os.path.exists(zip_path):
    print(f"Triggering direct browser download for: {zip_path}")
    files.download(zip_path)
else:
    print(f"Error: {zip_path} not found in the current directory.")

Triggering direct browser download for: patient_mortality_repository.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile README.md
# Patient Mortality Risk Prediction System

This repository contains a modular, production-grade Machine Learning pipeline designed to predict 30-day patient mortality risks based on clinical and physiological features. Built as a university-level engineering submission, it follows strict validation, testing, and modular separation of concerns.

## Repository Directory Structure

```text
├── src/
│   ├── ingest.py         # Handles CSV ingestion and missing path safety checks
│   ├── validate.py       # Excludes non-adult patients (<18 or >120) and physiologically invalid BP records
│   ├── features.py       # Engineers 'pulse_pressure' and 'comorbidity_count'
│   ├── train.py          # Splits data, trains a Random Forest model, and saves performance metrics
│   └── predict.py        # CLI script to make inference predictions on new patient samples
├── tests/
│   └── test_pipeline.py  # Automated unit testing suite executed via pytest
├── sample_inputs/
│   └── new_patients.csv  # Input layout template for prediction CLI
├── saved_model/
│   ├── model.pkl         # Serialized Random Forest classifier
│   └── evaluation_results.txt # Performance report containing ROC-AUC metric evaluation
├── requirements.txt      # Project system dependencies
└── README.md             # Project documentation
```

## Quick Start Guide

### 1. Installation
Ensure you have python installed, and run:
```bash
pip install -r requirements.txt
```

### 2. Model Training & Evaluation
To execute the entire clinical pipeline (loading, validating, engineering features, training the classifier, and exporting results):
```bash
python -m src.train
```

### 3. Running Unit Tests
Validate internal functions and rule metrics:
```bash
pytest tests/
```

### 4. Patient Inference CLI
To run predictions on a batch of new patients:
```bash
python src/predict.py --input sample_inputs/new_patients.csv

Writing README.md


In [ ]:
# Re-archive the project repository to ensure the newly created README.md is bundled
import zipfile
import os

def update_zip(zip_name="patient_mortality_repository.zip"):
    targets = [
        "src",
        "tests",
        "sample_inputs",
        "requirements.txt",
        "README.md"
    ]
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for target in targets:
            if os.path.exists(target):
                if os.path.isdir(target):
                    for root, dirs, files in os.walk(target):
                        for file in files:
                            filepath = os.path.join(root, file)
                            arcname = os.path.relpath(filepath, os.getcwd())
                            zipf.write(filepath, arcname)
                else:
                    zipf.write(target, target)
    print(f"Success! '{zip_name}' updated and verified containing README.md")

update_zip()

Success! 'patient_mortality_repository.zip' updated and verified containing README.md


In [ ]:
from IPython.display import display, Markdown
import os

readme_path = "README.md"
if os.path.exists(readme_path):
    with open(readme_path, "r") as f:
        content = f.read()
    display(Markdown(content))
else:
    print("README.md has not been generated yet. Please run the cell above first.")

# Patient Mortality Risk Prediction System

This repository contains a modular, production-grade Machine Learning pipeline designed to predict 30-day patient mortality risks based on clinical and physiological features. Built as a university-level engineering submission, it follows strict validation, testing, and modular separation of concerns.

## Repository Directory Structure

```text
├── src/
│   ├── ingest.py         # Handles CSV ingestion and missing path safety checks
│   ├── validate.py       # Excludes non-adult patients (<18 or >120) and physiologically invalid BP records
│   ├── features.py       # Engineers 'pulse_pressure' and 'comorbidity_count'
│   ├── train.py          # Splits data, trains a Random Forest model, and saves performance metrics
│   └── predict.py        # CLI script to make inference predictions on new patient samples
├── tests/
│   └── test_pipeline.py  # Automated unit testing suite executed via pytest
├── sample_inputs/
│   └── new_patients.csv  # Input layout template for prediction CLI
├── saved_model/
│   ├── model.pkl         # Serialized Random Forest classifier
│   └── evaluation_results.txt # Performance report containing ROC-AUC metric evaluation
├── requirements.txt      # Project system dependencies
└── README.md             # Project documentation
```

## Quick Start Guide

### 1. Installation
Ensure you have python installed, and run:
```bash
pip install -r requirements.txt
```

### 2. Model Training & Evaluation
To execute the entire clinical pipeline (loading, validating, engineering features, training the classifier, and exporting results):
```bash
python -m src.train
```

### 3. Running Unit Tests
Validate internal functions and rule metrics:
```bash
pytest tests/
```

### 4. Patient Inference CLI
To run predictions on a batch of new patients:
```bash
python src/predict.py --input sample_inputs/new_patients.csv


In [ ]:
from google.colab import files
import os

readme_path = "README.md"
if os.path.exists(readme_path):
    print(f"Triggering direct browser download for: {readme_path}")
    files.download(readme_path)
else:
    print(f"Error: {readme_path} not found. Please run the cell above to generate it.")

Triggering direct browser download for: README.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 1. Initialize Expanded Project Directory on Google Drive
We will structure the project under `/content/drive/MyDrive/AINT602_mortality_pipeline` to make it modular and clean.

In [ ]:
from pathlib import Path
import os
import shutil

PROJECT_ROOT = Path("/content/drive/MyDrive/AINT602_mortality_pipeline")

directories = [
    "data/raw",
    "data/processed",
    "data/rejected",
    "notebooks",
    "src",
    "tests",
    "sample_inputs",
    "saved_model",
    "reports",
    "outputs",
    ".github/workflows",
]

for directory in directories:
    (PROJECT_ROOT / directory).mkdir(parents=True, exist_ok=True)

print(f"Project created at: {PROJECT_ROOT}")

# Safely copy raw clinical dataset into our raw directory structure
source_csv = "/content/synthetic_clinical_dataset.csv"
dest_csv = PROJECT_ROOT / "data/raw/synthetic_clinical_dataset.csv"
if os.path.exists(source_csv):
    shutil.copy(source_csv, dest_csv)
    print(f"Copied raw dataset to: {dest_csv}")
else:
    print(f"Warning: {source_csv} not found in temporary runtime space.")

Project created at: /content/drive/MyDrive/AINT602_mortality_pipeline
Copied raw dataset to: /content/drive/MyDrive/AINT602_mortality_pipeline/data/raw/synthetic_clinical_dataset.csv


### 2. Move working directory into Project Root and install dependencies

In [ ]:
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")

# Install pinned dependencies
!pip install -q pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.2 joblib==1.4.2 matplotlib==3.8.0 seaborn==0.13.2 pyyaml==6.0.2 pytest==8.3.3

# Create requirements.txt file
requirements = """\
pandas==2.2.2
numpy==1.26.4
scikit-learn==1.5.2
joblib==1.4.2
matplotlib==3.8.0
seaborn==0.13.2
PyYAML==6.0.2
pytest==8.3.3
"""
(PROJECT_ROOT / "requirements.txt").write_text(requirements)
print("Saved requirements.txt with fully pinned versions.")

Current working directory: /content/drive/MyDrive/AINT602_mortality_pipeline
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> pandas

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Saved requirements.txt with fully pinned versions.


### 3. Create YAML Configuration File

In [ ]:
config_yaml = """
project:
  name: mortality-risk-mlops
  version: "1.0.0"
  random_seed: 42

data:
  raw_path: data/raw/synthetic_clinical_dataset.csv
  processed_path: data/processed/validated_patients.csv
  rejected_path: data/rejected/rejected_patients.csv

target:
  name: mortality

features:
  numeric:
    - age
    - bmi
    - systolic_bp
    - diastolic_bp
    - glucose
    - cholesterol
    - creatinine
    - diabetes
    - hypertension
    - pulse_pressure
    - comorbidity_count
  categorical:
    - sex
  excluded:
    - patient_id
    - mortality
    - readmission_30d
    - diagnosis

validation:
  minimum_age: 18
  maximum_age: 120
  allowed_sex:
    - Female
    - Male
    - Other

split:
  test_size: 0.15
  validation_size: 0.15

model:
  primary_metric: average_precision
  threshold_metric: f2
"""
(PROJECT_ROOT / "config.yaml").write_text(config_yaml)
print("Saved configuration parameters to config.yaml.")

Saved configuration parameters to config.yaml.


### 4. Create ingest.py and validate.py with complete execution paths

In [ ]:
ingest_code = '''
from pathlib import Path
import hashlib
import pandas as pd

REQUIRED_COLUMNS = [
    "patient_id",
    "age",
    "sex",
    "bmi",
    "systolic_bp",
    "diastolic_bp",
    "glucose",
    "cholesterol",
    "creatinine",
    "diabetes",
    "hypertension",
    "diagnosis",
    "readmission_30d",
    "mortality",
]

def calculate_sha256(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as file:
        for block in iter(lambda: file.read(65536), b""):
            sha256.update(block)
    return sha256.hexdigest()

def load_raw_data(file_path):
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Raw data file not found: {file_path}")
    data = pd.read_csv(file_path)
    missing_columns = sorted(set(REQUIRED_COLUMNS) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Required columns are missing: {missing_columns}")
    return data

if __name__ == "__main__":
    path = "data/raw/synthetic_clinical_dataset.csv"
    dataframe = load_raw_data(path)
    print(f"Rows: {len(dataframe):,}")
    print(f"Columns: {len(dataframe.columns)}")
    print(f"SHA-256: {calculate_sha256(path)}")
'''

(PROJECT_ROOT / "src/__init__.py").write_text("")
(PROJECT_ROOT / "src/ingest.py").write_text(ingest_code)
print("Ingestion system written successfully.")

Ingestion system written successfully.


In [ ]:
validation_code = '''
import json
from pathlib import Path
import pandas as pd
from src.ingest import load_raw_data

EXPECTED_BINARY = {
    "diabetes": {0, 1},
    "hypertension": {0, 1},
    "readmission_30d": {0, 1},
    "mortality": {0, 1},
}
EXPECTED_SEX = {"Female", "Male", "Other"}

def validate_categories(data):
    unexpected_sex = set(data["sex"].dropna().unique()) - EXPECTED_SEX
    if unexpected_sex:
        raise ValueError(f"Unexpected sex categories: {sorted(unexpected_sex)}")
    for column, allowed_values in EXPECTED_BINARY.items():
        invalid = set(data[column].dropna().unique()) - allowed_values
        if invalid:
            raise ValueError(f"{column} contains invalid values: {sorted(invalid)}")

def validate_data(data):
    duplicate_rows = int(data.duplicated().sum())
    duplicate_ids = int(data["patient_id"].duplicated().sum())
    missing_values = int(data.isna().sum().sum())

    validate_categories(data)

    adult_mask = data["age"].between(18, 120)
    valid_bp_mask = data["diastolic_bp"] < data["systolic_bp"]

    accepted_mask = adult_mask & valid_bp_mask
    accepted = data.loc[accepted_mask].copy()
    rejected = data.loc[~accepted_mask].copy()

    rejected["rejection_reason"] = ""
    rejected.loc[~rejected["age"].between(18, 120), "rejection_reason"] += "invalid_age;"
    rejected.loc[rejected["diastolic_bp"] >= rejected["systolic_bp"], "rejection_reason"] += "invalid_blood_pressure;"

    report = {
        "raw_rows": int(len(data)),
        "raw_columns": int(len(data.columns)),
        "missing_values": missing_values,
        "duplicate_rows": duplicate_rows,
        "duplicate_patient_ids": duplicate_ids,
        "under_18_or_over_120": int((~adult_mask).sum()),
        "invalid_blood_pressure": int((~valid_bp_mask & adult_mask).sum()),
        "accepted_rows": int(len(accepted)),
        "rejected_rows": int(len(rejected)),
        "accepted_mortality_positive": int(accepted["mortality"].sum()),
        "accepted_mortality_rate": float(accepted["mortality"].mean()),
    }
    return accepted, rejected, report

if __name__ == "__main__":
    raw = load_raw_data("data/raw/synthetic_clinical_dataset.csv")
    accepted, rejected, report = validate_data(raw)

    Path("data/processed").mkdir(parents=True, exist_ok=True)
    Path("data/rejected").mkdir(parents=True, exist_ok=True)
    Path("reports").mkdir(parents=True, exist_ok=True)

    accepted.to_csv("data/processed/validated_patients.csv", index=False)
    rejected.to_csv("data/rejected/rejected_patients.csv", index=False)

    with open("reports/data_validation.json", "w") as file:
        json.dump(report, file, indent=2)

    print(json.dumps(report, indent=2))
'''

(PROJECT_ROOT / "src/validate.py").write_text(validation_code)
print("Validation pipeline script written successfully.")

Validation pipeline script written successfully.


In [ ]:
# Execute ingestion and validation steps directly inside the pipeline
!python -m src.ingest
!python -m src.validate

Rows: 10,000
Columns: 14
SHA-256: bee99ea6e73b263a19402172cd61dc012089e1dccdd2ba13c43d12722e7a988b
{
  "raw_rows": 10000,
  "raw_columns": 14,
  "missing_values": 0,
  "duplicate_rows": 0,
  "duplicate_patient_ids": 0,
  "under_18_or_over_120": 204,
  "invalid_blood_pressure": 156,
  "accepted_rows": 9640,
  "rejected_rows": 360,
  "accepted_mortality_positive": 262,
  "accepted_mortality_rate": 0.027178423236514523
}


### 5. Create Engineered Features Script
We will define `ClinicalFeatureEngineer` as a reusable scikit-learn compatible transformer.

In [ ]:
features_code = '''
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class ClinicalFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        transformed = X.copy()
        transformed["pulse_pressure"] = (
            transformed["systolic_bp"]
            - transformed["diastolic_bp"]
        )
        transformed["comorbidity_count"] = (
            transformed["diabetes"].astype(int)
            + transformed["hypertension"].astype(int)
        )
        return transformed
'''

(PROJECT_ROOT / "src/features.py").write_text(features_code)
print("Feature engineering code written to src/features.py")

Feature engineering code written to src/features.py


### 6. Create Training and Evaluation Scripts
We will now write scripts to load configuration parameters, run stratified splits, compute custom metrics, and serialize the fitted pipelines.

In [ ]:
train_code = '''
import yaml
import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from src.ingest import load_raw_data
from src.validate import validate_data
from src.features import ClinicalFeatureEngineer

def build_pipeline(config, classifier):
    numeric_features = config["features"]["numeric"]
    categorical_features = config["features"]["categorical"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    full_pipeline = Pipeline(steps=[
        ("engineer", ClinicalFeatureEngineer()),
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    return full_pipeline

if __name__ == "__main__":
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)

    raw_df = load_raw_data(config["data"]["raw_path"])
    accepted, rejected, _ = validate_data(raw_df)

    X = accepted.drop(columns=config["features"]["excluded"], errors="ignore")
    y = accepted[config["target"]["name"]]

    # Split data stratified (70% Train, 15% Val, 15% Test)
    seed = config["project"]["random_seed"]
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=config["split"]["test_size"], random_seed=None, stratify=y, random_state=seed
    )

    val_rel_size = config["split"]["validation_size"] / (1.0 - config["split"]["test_size"])
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_rel_size, stratify=y_train_val, random_state=seed
    )

    print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

    # Baseline Pipeline
    baseline_model = DummyClassifier(strategy="prior")
    baseline_pipeline = build_pipeline(config, baseline_model)
    baseline_pipeline.fit(X_train, y_train)

    # Candidate Random Forest
    candidate_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight="balanced",
        random_state=seed
    )
    candidate_pipeline = build_pipeline(config, candidate_model)
    candidate_pipeline.fit(X_train, y_train)

    # Save candidate pipeline
    Path("saved_model").mkdir(exist_ok=True)
    joblib.dump(candidate_pipeline, "saved_model/model_pipeline.joblib")

    # Temporarily persist arrays for evaluation
    joblib.dump((baseline_pipeline, candidate_pipeline, X_val, y_val, X_test, y_test), "reports/eval_temp.joblib")
    print("Training process executed. Exported pipelines to joblib serialization files.")
'''
(PROJECT_ROOT / "src/train.py").write_text(train_code)
print("Training script written successfully.")

Training script written successfully.


In [ ]:
evaluate_code = '''
import json
import joblib
import yaml
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_recall_curve, auc, average_precision_score, recall_score,
    precision_score, fbeta_score, confusion_matrix, roc_auc_score, brier_score_loss
)

if __name__ == "__main__":
    baseline_pipeline, candidate_pipeline, X_val, y_val, X_test, y_test = joblib.load("reports/eval_temp.joblib")

    # Score on final test set
    probs = candidate_pipeline.predict_proba(X_test)[:, 1]
    preds = candidate_pipeline.predict(X_test)

    # Compute metrics
    pr_auc = average_precision_score(y_test, probs)
    rec = recall_score(y_test, preds)
    prec = precision_score(y_test, preds)
    f2 = fbeta_score(y_test, preds, beta=2)
    roc_auc = roc_auc_score(y_test, probs)
    brier = brier_score_loss(y_test, probs)

    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

    metrics = {
        "pr_auc": float(pr_auc),
        "recall": float(rec),
        "precision": float(prec),
        "f2_score": float(f2),
        "roc_auc": float(roc_auc),
        "brier_score": float(brier),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "true_negatives": int(tn),
        "true_positives": int(tp)
    }

    with open("reports/metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    # Generate and save metrics plot assets
    plt.figure(figsize=(6, 5))
    sns.heatmap([[tn, fp], [fn, tp]], annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig("reports/confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.close()

    # Precision-Recall Curve
    p, r, _ = precision_recall_curve(y_test, probs)
    plt.figure(figsize=(6, 5))
    plt.plot(r, p, label=f"Candidate (PR-AUC = {pr_auc:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.savefig("reports/precision_recall_curve.png", dpi=150, bbox_inches="tight")
    plt.close()

    print("Evaluation outputs computed and plots written to reports/.")
    print(json.dumps(metrics, indent=2))
'''
(PROJECT_ROOT / "src/evaluate.py").write_text(evaluate_code)
print("Evaluation script written successfully.")

Evaluation script written successfully.


In [ ]:
!python -m src.train
!python -m src.evaluate

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/drive/MyDrive/AINT602_mortality_pipeline/src/train.py", line 58, in <module>
    X_train_val, X_test, y_train_val, y_test = train_test_split(
                                               ~~~~~~~~~~~~~~~~^
        X, y, test_size=config["split"]["test_size"], random_seed=None, stratify=y, random_state=seed
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/sklearn/utils/_param_validation.py", line 194, in wrapper
    params = func_sig.bind(*args, **kwargs)
  File "/usr/lib/python3.13/inspect.py", line 3316, in bind
    return self._bind(args, kwargs)
           ~~~~~~~~~~^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/inspect.py", line 3305, in _bind
    raise TypeError(
        'got an unexpected keyword argument {arg!r}'.for

### 7. Create Prediction Script and Model Card
We will write `predict.py` using command line arguments to make inferences on test clinical rows, and save metadata to a clean `model_card.md` file.

In [ ]:
predict_code = '''
import argparse
import joblib
import pandas as pd
from pathlib import Path
from datetime import datetime

def run_prediction(input_csv, output_csv="outputs/predictions.csv"):
    model_path = "saved_model/model_pipeline.joblib"
    if not Path(model_path).exists():
        raise FileNotFoundError(f"Model pipeline not found: {model_path}")

    pipeline = joblib.load(model_path)
    df = pd.read_csv(input_csv)
    original_df = df.copy()

    probs = pipeline.predict_proba(df)[:, 1]
    preds = pipeline.predict(df)

    original_df["mortality_probability"] = probs
    original_df["mortality_flag"] = preds
    original_df["model_version"] = "1.0.0"
    original_df["prediction_timestamp"] = datetime.now().isoformat()

    Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
    original_df[["patient_id", "mortality_probability", "mortality_flag", "model_version", "prediction_timestamp"]].to_csv(output_csv, index=False)
    print(f"Predictions exported successfully to {output_csv}")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=str, required=True)
    parser.add_argument("--output", type=str, default="outputs/predictions.csv")
    args = parser.parse_args()
    run_prediction(args.input, args.output)
'''
(PROJECT_ROOT / "src/predict.py").write_text(predict_code)
print("Prediction script written successfully.")

Prediction script written successfully.


In [ ]:
model_card = """
# Model Card - Clinical Mortality Risk Classifier

- **Model Type**: Random Forest Classifier with class weight balanced adjustments
- **Version**: 1.0.0
- **Intended Use**: Academic demonstration of an MLOps validation and scoring pipeline
- **Features Used**: age, bmi, systolic_bp, diastolic_bp, glucose, cholesterol, creatinine, diabetes, hypertension, pulse_pressure, comorbidity_count, sex
- **Target variable**: Recorded clinical dataset mortality outcome
"""
(PROJECT_ROOT / "saved_model/model_card.md").write_text(model_card)
print("Created saved_model/model_card.md")

Created saved_model/model_card.md


### 8. Create Automated Pipeline Tests
We will write `tests/test_pipeline.py` containing test assertions.

In [ ]:
test_code = '''
import pandas as pd
import pytest
from src.features import ClinicalFeatureEngineer
from src.validate import validate_data

def valid_patient():
    return {
        "patient_id": 1,
        "age": 65,
        "sex": "Female",
        "bmi": 27.5,
        "systolic_bp": 130,
        "diastolic_bp": 80,
        "glucose": 105,
        "cholesterol": 190,
        "creatinine": 1.0,
        "diabetes": 1,
        "hypertension": 1,
        "diagnosis": "Normal",
        "readmission_30d": 0,
        "mortality": 0,
    }

def test_feature_engineering():
    data = pd.DataFrame([valid_patient()])
    result = ClinicalFeatureEngineer().fit_transform(data)
    assert result.loc[0, "pulse_pressure"] == 50
    assert result.loc[0, "comorbidity_count"] == 2

def test_underage_patient_is_rejected():
    patient = valid_patient()
    patient["age"] = 17
    accepted, rejected, report = validate_data(pd.DataFrame([patient]))
    assert len(accepted) == 0
    assert len(rejected) == 1

def test_invalid_blood_pressure_is_rejected():
    patient = valid_patient()
    patient["systolic_bp"] = 70
    patient["diastolic_bp"] = 90
    accepted, rejected, report = validate_data(pd.DataFrame([patient]))
    assert len(accepted) == 0
    assert "invalid_blood_pressure" in rejected.iloc[0]["rejection_reason"]
'''
(PROJECT_ROOT / "tests/test_pipeline.py").write_text(test_code)
print("Saved unit tests script.")

Saved unit tests script.


In [ ]:
# Execute batch tests and prediction outputs
!python -m pytest -v
!python -m src.predict --input sample_inputs/new_patients.csv --output outputs/predictions.csv
!cat outputs/predictions.csv

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/AINT602_mortality_pipeline
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 3 items                                                              

tests/test_pipeline.py::test_feature_engineering PASSED                  [ 33%]
tests/test_pipeline.py::test_underage_patient_is_rejected PASSED         [ 66%]
tests/test_pipeline.py::test_invalid_blood_pressure_is_rejected PASSED   [100%]

============================== 3 passed in 2.75s ===============================
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/drive/MyDrive/AINT602_mortality_pipeline/src/predict.py", line 34, in <module>
    run_prediction(args.input, args.o

### 9. Final Packaging & Automatic Direct Download
We will package the entire repository as a clean zip file and download it along with the README.md file.

In [ ]:
import zipfile
import os
from google.colab import files

def create_bundle(zip_name="AINT602_mortality_pipeline.zip"):
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, filenames in os.walk(PROJECT_ROOT):
            # Exclude cache or temp storage items
            if ".ipynb_checkpoints" in root:
                continue
            for filename in filenames:
                filepath = os.path.join(root, filename)
                arcname = os.path.relpath(filepath, PROJECT_ROOT)
                zipf.write(filepath, arcname)
    print(f"Bundle created: {zip_name}")
    files.download(zip_name)

create_bundle()

Bundle created: AINT602_mortality_pipeline.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%%writefile src/train.py
import yaml
import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from src.ingest import load_raw_data
from src.validate import validate_data
from src.features import ClinicalFeatureEngineer

def build_pipeline(config, classifier):
    numeric_features = config["features"]["numeric"]
    categorical_features = config["features"]["categorical"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    full_pipeline = Pipeline(steps=[
        ("engineer", ClinicalFeatureEngineer()),
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])
    return full_pipeline

if __name__ == "__main__":
    with open("config.yaml", "r") as f:
        config = yaml.safe_load(f)

    raw_df = load_raw_data(config["data"]["raw_path"])
    accepted, rejected, _ = validate_data(raw_df)

    X = accepted.drop(columns=config["features"]["excluded"], errors="ignore")
    y = accepted[config["target"]["name"]]

    seed = config["project"]["random_seed"]
    # Split data stratified (70% Train, 15% Val, 15% Test)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=config["split"]["test_size"], stratify=y, random_state=seed
    )

    val_rel_size = config["split"]["validation_size"] / (1.0 - config["split"]["test_size"])
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=val_rel_size, stratify=y_train_val, random_state=seed
    )

    print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

    # 1. Baseline Prior Dummy Pipeline
    baseline_model = DummyClassifier(strategy="prior")
    baseline_pipeline = build_pipeline(config, baseline_model)
    baseline_pipeline.fit(X_train, y_train)

    # 2. Logistic Regression Candidate Pipeline
    lr_model = LogisticRegression(class_weight="balanced", random_state=seed, max_iter=1000)
    lr_pipeline = build_pipeline(config, lr_model)
    lr_pipeline.fit(X_train, y_train)

    # 3. Random Forest Candidate Pipeline
    rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight="balanced", random_state=seed)
    rf_pipeline = build_pipeline(config, rf_model)
    rf_pipeline.fit(X_train, y_train)

    # Save winning Random Forest pipeline
    Path("saved_model").mkdir(exist_ok=True)
    joblib.dump(rf_pipeline, "saved_model/model_pipeline.joblib")

    # Temporarily persist models and arrays for evaluation comparisons
    joblib.dump((baseline_pipeline, lr_pipeline, rf_pipeline, X_val, y_val, X_test, y_test), "reports/eval_temp.joblib")
    print("Training process executed. Exported pipelines to joblib serialization files.")

Overwriting src/train.py


In [ ]:
# Run training pipeline
!python -m src.train

Train size: 6748, Val size: 1446, Test size: 1446
Training process executed. Exported pipelines to joblib serialization files.


In [ ]:
# Run evaluation module to generate metric outputs and comparison curves
!python -m src.evaluate

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/drive/MyDrive/AINT602_mortality_pipeline/src/evaluate.py", line 14, in <module>
    baseline_pipeline, candidate_pipeline, X_val, y_val, X_test, y_test = joblib.load("reports/eval_temp.joblib")
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: too many values to unpack (expected 6)


In [ ]:
import json
import os

metrics_path = "reports/metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path, "r") as f:
        metrics = json.load(f)
    import pandas as pd
    display(pd.Series(metrics).to_frame("Metric Value"))
else:
    print(f"File not found at: {metrics_path}. Please execute cell 'e6e69de5' first!")

File not found at: reports/metrics.json. Please execute cell 'e6e69de5' first!


In [ ]:
from google.colab import drive
import os
from pathlib import Path

print("=== Mounting Google Drive ===")
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = Path("/content/drive/MyDrive/AINT602_mortality_pipeline")

def list_project_structure(path, indent=""):
    path = Path(path)
    for item in sorted(path.iterdir()):
        if item.name.startswith(".") or item.name == "__pycache__":
            continue
        if item.is_dir():
            print(f"{indent}📁 {item.name}/")
            list_project_structure(item, indent + "  ")
        else:
            print(f"{indent}📄 {item.name}")

if PROJECT_ROOT.exists():
    print(f"\nSUCCESS: Project root found at {PROJECT_ROOT}\n")
    os.chdir(PROJECT_ROOT)
    print("=== Project Directory Listing ===")
    list_project_structure(".")
else:
    print(f"\nERROR: Project root NOT found at {PROJECT_ROOT}")

=== Mounting Google Drive ===
Mounted at /content/drive

SUCCESS: Project root found at /content/drive/MyDrive/AINT602_mortality_pipeline

=== Project Directory Listing ===
📄 AINT602_mortality_pipeline.zip
📄 config.yaml
📁 data/
  📁 processed/
    📄 validated_patients.csv
  📁 raw/
    📄 synthetic_clinical_dataset.csv
  📁 rejected/
    📄 rejected_patients.csv
📁 notebooks/
📁 outputs/
📁 reports/
  📄 data_validation.json
  📄 eval_temp.joblib
📄 requirements.txt
📁 sample_inputs/
📁 saved_model/
  📄 model_card.md
  📄 model_pipeline.joblib
📁 src/
  📄 __init__.py
  📄 evaluate.py
  📄 features.py
  📄 ingest.py
  📄 predict.py
  📄 train.py
  📄 validate.py
📁 tests/
  📄 test_pipeline.py


In [ ]:
# Run unit testing suite directly on the test_pipeline module
!python -m pytest tests/test_pipeline.py

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/drive/MyDrive/AINT602_mortality_pipeline
plugins: langsmith-0.14.17, typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3
collected 3 items                                                              

tests/test_pipeline.py ...                                               [100%]

============================== 3 passed in 1.29s ===============================


In [ ]:
# Cell deactivated: Training is now complete and corrected in cell f79f1d16.